# MedBoard — Train Attention U-Net Segmentation Model

**Run this notebook in Google Colab to train the Attention U-Net.**

Attention U-Net (Oktay et al., 2018) enhances standard U-Net by adding **Attention Gates (AGs)** to each skip connection:
- Uses coarse decoder gating signals to filter out irrelevant background MRI tissue and noise.
- Highlights salient tumor boundaries before concatenation into decoder stages.
- Adds only **~1.68% parameter overhead** (+132k parameters) over standard U-Net.

### Pipeline Overview:
1. **Environment Setup** (Mount Drive, pull repo, prepare dataset on local SSD)
2. **Verify GPU** (Colab T4)
3. **Load BRISC 2025 Dataset** (Train, Validation, Test splits)
4. **Build Attention U-Net** (Shape and parameter count validation)
5. **Train** with Dice + BCE loss, FP16 mixed precision, and early stopping
6. **Save Best Checkpoint & Final Weights** to Drive
7. **Plot Training Curves** (Loss & Dice)
8. **Evaluate on Test Set** (Dice + IoU metrics)
9. **Visualize Predictions & Attention Gate Heatmaps**

**Expected training time:** ~25–35 minutes on Colab T4 GPU

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import shutil, os, sys

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code)"}')

if IS_COLAB:
    from google.colab import drive

    # Mount Google Drive
    drive.mount('/content/drive')

    # Clone or update the MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # Copy dataset from Drive to local SSD (faster I/O during training)
    if not os.path.exists('/content/data'):
        print('Copying dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Done.')
    else:
        print('Dataset already on local SSD.')

    # Install dependencies
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    # Set Python path so we can import from MedBoard
    sys.path.insert(0, '/content/MedBoard')

    # Colab paths
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'
    CONFIG_PATH = '/content/MedBoard/configs/config.yaml'

else:
    # Local / VS Code paths
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'weights')
    CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'config.yaml')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

# ── Output weights paths ──────────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'attention_unet_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'attention_unet_brisc.pth')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'\nSetup complete!')
print(f'  Data       → {DATA_ROOT}')
print(f'  Weights    → {WEIGHTS_DIR}')
print(f'  Checkpoint → {CHECKPOINT_PATH}')
print(f'  Final      → {FINAL_PATH}')

## Cell 2 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Please select Runtime → Change runtime type → T4 GPU')

## Cell 3 — Load Dataset

In [ ]:
from modules.seg_dataset import get_seg_dataloaders

train_loader, val_loader, test_loader = get_seg_dataloaders(
    train_images_dir = f'{DATA_ROOT}/segmentation_task/train/images',
    train_masks_dir  = f'{DATA_ROOT}/segmentation_task/train/masks',
    test_images_dir  = f'{DATA_ROOT}/segmentation_task/test/images',
    test_masks_dir   = f'{DATA_ROOT}/segmentation_task/test/masks',
    batch_size  = 8,
    num_workers = 2,
    val_split   = 0.1,
    config_path = CONFIG_PATH,
)

print(f'DataLoaders ready:')
print(f'  Train: {len(train_loader.dataset)} samples → {len(train_loader)} batches')
print(f'  Val:   {len(val_loader.dataset)} samples → {len(val_loader)} batches')
print(f'  Test:  {len(test_loader.dataset)} samples → {len(test_loader)} batches')

## Cell 4 — Build Attention U-Net Model

In [ ]:
import yaml
from modules.unet import build_attention_unet, UNet, count_parameters

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

seg_cfg = cfg.get('segmentation', {})
base_features = 32  # Standard baseline U-Net size (7.85M std vs 7.98M attention)

# Build Attention U-Net
model = build_attention_unet({
    'in_channels': seg_cfg.get('in_channels', 3),
    'out_channels': seg_cfg.get('out_channels', 1),
    'base_features': base_features,
})

# Compare parameter count with standard U-Net
std_model = UNet(in_channels=3, out_channels=1, base_features=base_features)
std_params = count_parameters(std_model)
attn_params = count_parameters(model)
overhead = (attn_params - std_params)

print(f'Attention U-Net Parameters: {attn_params:,} ({attn_params/1e6:.2f}M)')
print(f'Standard U-Net Parameters:  {std_params:,} ({std_params/1e6:.2f}M)')
print(f'Attention Gates Overhead:   +{overhead:,} (+{(overhead/std_params)*100:.2f}%)')

# Forward shape validation
dummy = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    out = model(dummy)
print(f'\nShape Check: Input {tuple(dummy.shape)} → Output {tuple(out.shape)}')
assert out.shape == (1, 1, 256, 256), 'Output shape mismatch!'
print('Shape check passed successfully!')

## Cell 5 — Train Model

In [ ]:
from modules.seg_trainer import train_segmentation

history = train_segmentation(
    model             = model,
    train_loader      = train_loader,
    val_loader        = val_loader,
    epochs            = 40,
    learning_rate     = 2e-4,
    patience          = 10,
    lr_scheduler_type = 'cosine',
    checkpoint_path   = CHECKPOINT_PATH,
    device_str        = 'cuda',
)

## Cell 6 — Save Final Weights to Drive

In [ ]:
import shutil

# Copy best checkpoint to final weights path
shutil.copy(CHECKPOINT_PATH, FINAL_PATH)
print(f'Final weights saved to: {FINAL_PATH}')
print(f'Best val dice: {max(history["val_dice"]):.4f}  ({max(history["val_dice"])*100:.2f}%)')

## Cell 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', color='royalblue', linewidth=2)
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
ax1.set_title('Loss per Epoch (Dice + BCE)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice
ax2.plot(epochs_ran, history['train_dice'], label='Train Dice', color='royalblue', linewidth=2)
ax2.plot(epochs_ran, history['val_dice'],   label='Val Dice',   color='tomato',    linewidth=2)
ax2.axhline(0.75, color='green', linestyle='--', alpha=0.5, label='Good (0.75)')
ax2.axhline(0.85, color='gold',  linestyle='--', alpha=0.5, label='Excellent (0.85)')
ax2.set_title('Dice Score per Epoch', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice')
ax2.set_ylim([0, 1])
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('Attention U-Net Training — MedBoard', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = f'{WEIGHTS_DIR}/attention_unet_training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved to {plot_path}')

## Cell 8 — Evaluate on Test Set

In [ ]:
from modules.seg_trainer import validate_one_epoch, DiceBCELoss
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'model' not in locals():
    model = build_attention_unet(cfg.get('segmentation', {}))
    model.load_state_dict(torch.load(FINAL_PATH, map_location=device))
    print(f'Model loaded from: {FINAL_PATH}')

model = model.to(device)
model.eval()

criterion = DiceBCELoss()
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)

print('=' * 65)
print('        Attention U-Net — Test Set Evaluation')
print('=' * 65)
print(f'  Test Loss (Dice+BCE): {test_loss:.4f}')
print(f'  Test Dice Score:      {test_dice:.4f} ({test_dice*100:.2f}%)')
print('=' * 65)

## Cell 9 — Visualize Test Predictions and Attention Gate Heatmaps

In [ ]:
import random
import matplotlib.pyplot as plt

test_dataset = test_loader.dataset
indices = random.sample(range(len(test_dataset)), 4)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Attention U-Net — Predictions & Attention Gate Maps', fontsize=16, fontweight='bold')

for row, idx in enumerate(indices):
    img_tensor, mask_tensor = test_dataset[idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_input)
        pred = torch.sigmoid(logits).squeeze().cpu().numpy()
        # Extract attention map from dec1 (highest resolution AG)
        attn_map = model.dec1.attn.last_alpha.squeeze().cpu().numpy()

    # De-normalize image for plotting
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    gt_mask = mask_tensor.squeeze().cpu().numpy()

    # MRI input
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('MRI Input')
    axes[row, 0].axis('off')

    # Ground truth
    axes[row, 1].imshow(gt_mask, cmap='gray')
    axes[row, 1].set_title('Ground Truth')
    axes[row, 1].axis('off')

    # Predicted mask
    axes[row, 2].imshow(pred > 0.5, cmap='hot')
    axes[row, 2].set_title(f'Prediction (Prob max: {pred.max():.2f})')
    axes[row, 2].axis('off')

    # Attention map (alpha)
    axes[row, 3].imshow(attn_map, cmap='magma')
    axes[row, 3].set_title('Dec1 Attention Gate (α)')
    axes[row, 3].axis('off')

plt.tight_layout()
plt.show()